# Quickstart - Bali

End-to-end bias correction of IMERG Late Run V07 against CPC-UNI, over Bali, for a single dekad.

Three correction stages are produced: **LS**, **LSEQM**, and **LSEQM+DL**. Total runtime on Colab CPU: ~3 minutes.

For the methodology, see the [Methodology Overview](https://bennyistanto.github.io/hybrid-bias-correction/methodology/) in the documentation.

---

## 1. Setup

Clone the repository and add it to the Python path. The Bali example data ships inside the repo (~11 MB), so this is the only setup step.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/bennyistanto/hybrid-bias-correction.git'
REPO_DIR = '/content/hybrid-bias-correction'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Lightweight extras Colab does not preinstall.
subprocess.run(['pip', 'install', '-q', 'cartopy', 'netCDF4'], check=False)

print('Repo:', REPO_DIR)
print('CWD:', os.getcwd())

## 2. Load the Bali config

`config_bali.yml` points at `data/example_bali/` and uses the same parameters as the operational `config.yml` (so Bali results are directly comparable to the full-Indonesia run).

In [ ]:
from src.config import initialize_config

config = initialize_config('config_bali.yml')
print(f'AOI lat:   {config.aoi["lat_range"]}')
print(f'AOI lon:   {config.aoi["lon_range"]}')
print(f'Input dir: {config.input_dir}')
print(f'Output:    {config.output_dir}')

## 3. Load and align the grids

IMERG (ascending lat) and CPC (descending lat) are reindexed onto a common grid, and the land/sea mask is applied so ocean cells become NaN.

In [ ]:
from src.io import load_imerg, load_cpc, align_grids

imerg = load_imerg(config.imergl_file)
cpc   = load_cpc(config.cpc_file)
imerg, cpc = align_grids(imerg, cpc, mask_file=config.mask_file)

print('IMERG:', dict(imerg.sizes))
print('CPC  :', dict(cpc.sizes))

## 4. Run the pipeline for one dekad

This single call produces LS, LSEQM, and LSEQM+DL outputs for the chosen dekad:

1. Linear Scaling - matches the long-term mean.
2. Empirical Quantile Mapping with a GPD tail - reshapes the full distribution.
3. CNN refinement - learns the residual spatial bias.
4. Station-density blending - tempers CNN influence in gauge-sparse cells.

In [ ]:
from src.bias_correction import run_correction_pipeline

MONTH, DEKAD = 3, 1   # March, dekad 1 (days 1-10)

run_correction_pipeline(imerg, cpc, month=MONTH, dekad=DEKAD)

## 5. Inspect the outputs

In [ ]:
from pathlib import Path
out = Path(config.output_dir)
for stage in ['corrected_ls', 'corrected_lseqm', 'corrected_lseqmdl', 'trained_models']:
    files = sorted((out / stage).glob('*'))
    print(f'{stage:24s}  ({len(files)} files)')
    for f in files:
        print('   ', f.name)

## 6. Side-by-side comparison

Plot raw IMERG-L, CPC reference, and the three corrected stages for the chosen dekad.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

prefix = config.general['filename_prefix']
files = {
    'IMERG-L (raw)': config.imergl_file,
    'CPC reference': config.cpc_file,
    'LS':       out / f'corrected_ls/{prefix}_ls_corrected_imergl_month{MONTH:02d}_dekad{DEKAD}.nc4',
    'LSEQM':    out / f'corrected_lseqm/{prefix}_lseqm_corrected_imergl_month{MONTH:02d}_dekad{DEKAD}.nc4',
    'LSEQM+DL': out / f'corrected_lseqmdl/{prefix}_lseqmdl_corrected_imergl_month{MONTH:02d}_dekad{DEKAD}.nc4',
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)
for ax, (label, path) in zip(axes, files.items()):
    ds = xr.open_dataset(path)
    var = 'precip' if 'precip' in ds.data_vars else 'precipitation'
    field = ds[var].mean('time') if 'time' in ds[var].dims else ds[var]
    field.plot(ax=ax, cmap='Blues', vmin=0, vmax=20, cbar_kwargs={'label': 'mm/day'})
    ax.set_title(label)
plt.tight_layout()
plt.show()

## Next steps

- Read the [QA Framework](https://bennyistanto.github.io/hybrid-bias-correction/tutorials/qa-framework.html) tutorial to score the corrections.
- Open [`02_lseqmdl_bias_correction.ipynb`](02_lseqmdl_bias_correction.ipynb) to run all 36 dekads.
- Adapt `config_bali.yml` to another small AOI (see [AOI Setup](https://bennyistanto.github.io/hybrid-bias-correction/user-guide/aoi-setup.html)).